# __Project: Mandarin ASR with wav2vec 2.0 and CTC__

- This project follows the general workflow of the Hugging Face tutorial [Fine-Tuning XLS-R for Multilingual ASR with 🤗 Transformers](https://huggingface.co/blog/fine-tune-xlsr-wav2vec2). 

- The original tutorial has been adapted to the current Hugging Face Transformers API and to Mandarin speech recognition using the AISHELL-3 corpus.

- The goal of the project is to fine-tune a pretrained XLS-R model for Mandarin automatic speech recognition (ASR) using Connectionist Temporal Classification (CTC). Two output representations are investigated:

1. Chinese characters (Hanzi)
2. Tone-marked Pinyin syllables

- This notebook covers the third part of the project, including building data collator and fine-tuning the model.

- _AI-assisted development was used for code explanation, debugging, and API migration. Python version 3.13.14, Transformers version 5.12.1_

In [1]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

root = Path(os.environ["LOCALAPPDATA"]) / "ffmpeg-shared"
avcodec = next(root.rglob("avcodec-*.dll"), None)
if avcodec is None:
    raise FileNotFoundError(f"找不到 FFmpeg DLL：{root}")

ffmpeg_bin = avcodec.parent
os.environ["PATH"] = str(ffmpeg_bin) + os.pathsep + os.environ["PATH"]
ffmpeg_dll_dir = os.add_dll_directory(str(ffmpeg_bin))

import torch

print("Python:", sys.executable)
print("FFmpeg DLL:", avcodec)
print("PyTorch:", torch.__version__, "| CUDA 可用:", torch.cuda.is_available())
print("TorchCodec:", version("torchcodec"))

from torchcodec.decoders import AudioDecoder
print("TorchCodec 导入成功")

Python: c:\Users\liudr\anaconda3\envs\liushp02\python.exe
FFmpeg DLL: C:\Users\liudr\AppData\Local\ffmpeg-shared\ffmpeg-9.1-full_build-shared\bin\avcodec-63.dll
PyTorch: 2.15.0+cu132 | CUDA 可用: True
TorchCodec: 0.17.0+cu132
TorchCodec 导入成功


In [2]:
import torchcodec

In [3]:
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any

import numpy as np
import torch

from datasets import Audio, load_from_disk
from jiwer import cer, wer
from transformers import (
    Trainer,
    TrainingArguments,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    set_seed,
)

c:\Users\liudr\anaconda3\envs\liushp02\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
PROJECT_ROOT = Path.cwd()

PROCESSED_DATA_DIR = (PROJECT_ROOT / "processed_data")

PROCESSOR_DIR = (PROJECT_ROOT / "processors")

MODEL_OUTPUT_DIR = (PROJECT_ROOT / "model_outputs")

In [5]:
dataset = load_from_disk(PROCESSED_DATA_DIR / "aishell3_dataset")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio_path', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio_path', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio_path', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
{'id': 'SSB03820023', 'speaker_id': 'SSB0382', 'audio_path': 'aishell3/train/wav/SSB0382/SSB03820023.wav', 'hanzi': '但这一点已经足够表明卷福的超高人气', 'pinyin': 'dan4 zhe4 yi4 dian2 yi3 jing1 zu2 gou4 biao3 ming2 juan4 fu2 de5 chao1 gao1 ren2 qi4', 'age_group': 'B', 'gender': 'female', 'accent': 'north'}


In [6]:
# choose here the tasks
TASK = "hanzi"
# TASK = "pinyin"
# TASK = "letter"

In [7]:
class PinyinSyllableCTCTokenizer(
    Wav2Vec2CTCTokenizer):
    def prepare_for_tokenization(self, text, **kwargs):
        text = "|".join(text.lower().split())
        return text, kwargs

In [8]:
# after choosing the task, we can set the processor and model paths accordingly
# also we define here the metrics that used in the evaluation. For Hanzi and Pinyin Letter we use CER, and for Pinyin Syllable we use WER
if TASK == "hanzi":
    processor = (Wav2Vec2Processor.from_pretrained(PROCESSOR_DIR / "hanzi"))
    text_column = "hanzi"
    metric_name = "cer"

elif TASK == "letter":
    processor = (Wav2Vec2Processor.from_pretrained(PROCESSOR_DIR / "letter"))
    text_column = "pinyin"
    metric_name = "cer"

elif TASK == "pinyin":
    pinyin_dir = (PROCESSOR_DIR / "pinyin")

    pinyin_tokenizer = (PinyinSyllableCTCTokenizer.from_pretrained(pinyin_dir))
    feature_extractor = (Wav2Vec2FeatureExtractor.from_pretrained(pinyin_dir))
    processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=pinyin_tokenizer)

    text_column = "pinyin"
    metric_name = "wer"

else:
    raise ValueError(f"Unsupported task: {TASK}")

print(f"Task: {TASK} | Text column: {text_column} | Metric: {metric_name}")
print("Vocab size:", len(processor.tokenizer))

Task: hanzi | Text column: hanzi | Metric: cer
Vocab size: 4017


#### Inference Time Data Preparation

In [9]:
def resolve_audio_paths(batch):
    return {
        "audio": [
            str(PROJECT_ROOT / relative_path)
            for relative_path
            in batch["audio_path"]
        ]
    }

# {
#     "audio_path": "aishell3/train/wav/SSB0005/SSB00050001.wav",
#     "audio": "/Users/.../aishell3/train/wav/SSB0005/SSB00050001.wav",
#     "hanzi": "...",
#     "pinyin": "...",
# }

In [10]:
dataset = dataset.map(
    resolve_audio_paths,
    batched=True,
    desc="Resolving audio paths",
)

In [11]:
dataset = (
    dataset.cast_column(
        "audio",
        Audio(
            sampling_rate=16_000,
            num_channels=1,
        )
    )
)

# dataset = dataset.cast_column(
#     "audio",
#     Audio(
#         sampling_rate=processor.feature_extractor.sampling_rate,
#         num_channels=1,
#     ),
# )

In [12]:
def make_training_dataset(raw_dataset, processor, text_column):
    sampling_rate = processor.feature_extractor.sampling_rate

    def prepare_batch(batch):
        samples = [audio.get_all_samples() for audio in batch["audio"]]

        processed = processor(
            audio=[sample.data.squeeze(0).numpy() for sample in samples],
            sampling_rate=sampling_rate,
            text=batch[text_column],
        )

        return {
            "input_values": processed["input_values"],
            "input_length": [len(values) for values in processed["input_values"]],
            "labels": processed["labels"],
        }

    return raw_dataset.with_transform(
        prepare_batch,
        columns=["audio", text_column],
    )


training_dataset = make_training_dataset(dataset, processor, text_column)
#{
#     "id": "SSB00050001",
#     "audio_path": {...},
#     "text": "广州女大学生……",
#     "input_values": [...],
#     "input_length": 123456,
#     "labels": [...]
# }

In [13]:
test_example = training_dataset["train"][0]

print("Input length:", test_example["input_length"])
print("Number of labels:", len(test_example["labels"]))
print("True text:", dataset["train"][text_column][0])
print("First input values:", test_example["input_values"][:10])
print("Labels:", test_example["labels"])

Input length: 78809
Number of labels: 17
True text: 但这一点已经足够表明卷福的超高人气
First input values: [ 0.04401884  0.02261921  0.00648973  0.03158857  0.03285171 -0.01215215
 -0.04094664 -0.03844285 -0.00987181  0.02702106]
Labels: [151, 3512, 0, 2161, 1053, 2739, 3418, 775, 3199, 1600, 440, 2560, 2410, 3413, 3936, 93, 1879]


#### __📊 Data Collator__: Final preparation before fine-tuning

__What does the Data Collator contain?__
```
DataCollatorCTCWithPadding class
├── processor = hanzi_processor  → Hanzi collator
├── processor = pinyin_processor → Pinyin-syllable collator
└── processor = phone_processor  → Pinyin-character collator
```

__Why do we need Data Collator?__

- The data collator combines a list of individual examples into one batch. It dynamically pads the audio inputs and text labels separately, creates an attention mask for the padded audio, replaces label-padding positions with -100, and returns the tensors that the Trainer passes to the model.

__Basic structure of Data Collator?__
```
Dataset
   ↓
DataLoader selects several examples
   ↓
Data Collator creates a padded batch
   ↓
Trainer passes the batch to the model
   ↓
Model computes CTC loss
```

In [14]:
@dataclass
class DataCollatorCTCWithPadding:
    processor : Wav2Vec2Processor
    padding: bool | str = True

    '''
    Here the only args we give from the Trainer to the Data Collator is the feature of the input.

    features shaped like:
        features = [
            {
                "input_values": [...],
                "labels": [...],
            }]

    In the end, data collator will give back the batches in shape: 
    batch = {
        "input_values": torch.Tensor(...),
        "attention_mask": torch.Tensor(...),
        "labels": torch.Tensor(...),
    }

    One Example Input Contents:
    Input length: 78809
    Number of labels: 17
    true labels: 但这一点已经足够表明卷福的超高人气
    First input values: [ 0.04401884  0.02261921  0.00648973  0.03158857  0.03285171 -0.01215215
    -0.04094664 -0.03844285 -0.00987181  0.02702106]
    Labels: [151, 3512, 0, 2161, 1053, 2739, 3418, 775, 3199, 1600, 440, 2560, 2410, 3413, 3936, 93, 1879]

'''

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        # step 1: the data collator must get the audio input
        # input_values: the extracted features in single audiop file, e.g mel or mfcc
        input_features = [{"input_values": feature["input_values"]} for feature in features]

        # step 2: prepare the label data, then combine these two features together
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # step 3: padding the audio and the label
        # padding the audio using the feature extractor, audio padding using 0.0
        batch = (self.processor.feature_extractor.pad(input_features, padding = self.padding, return_tensors= "pt"))

        # padding labels using the tokenizer, label padding using -100
        labels_batch = (self.processor.tokenizer.pad(label_features, padding = self.padding, return_tensors = "pt"))

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        batch["labels"] = labels

        return batch

In [15]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

batch = data_collator([test_example, training_dataset["train"][1]])

assert batch["input_values"].ndim == 2
assert batch["attention_mask"].shape == batch["input_values"].shape
assert batch["labels"].ndim == 2
assert batch["input_values"].shape[0] == batch["labels"].shape[0] == 2

for key, tensor in batch.items():
    print(f"{key} shape: {tensor.shape}")
print(f"{TASK} data collator test passed.")

input_values shape: torch.Size([2, 78809])
attention_mask shape: torch.Size([2, 78809])
labels shape: torch.Size([2, 17])
hanzi data collator test passed.


### __🏃‍♀️‍➡️Define the Trainer, and fine-tuning the XLR-wave2vec__
- We first fine-tune and validate the complete training process for the Hanzi recognition task.


#### __✔️ Define the Evaluation Metrics__
- for the Hanzi tokenization, we use Character Error Rate

- for the Pinyin tokenization, we use Word Error Rate

- for the Phone tokenization, we use the Edit Distance

Issues needs to be fixed: 
1. Phone edit distance needs to be normalized
2. Choose different metrics based on different metrics, but if we want to compare the model abilities variants under different settings, need to be normalized?
3. Levenshtein edit distance needs Chinese phone pair (sh to zh as a pair to be computed)
4. Pinyin WER actually is the syllable error rate

Evaluation Metrics
- Hanzi uses character error rate (CER).
- Pinyin letters use CER with spaces removed. Tone numbers count as characters.
- Pinyin syllables use word error rate (WER) on space-separated syllables.
Metrics use the original transcripts as references so that characters and syllables outside the training vocabulary retain their original spelling.

In [16]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]

    prediction_ids = logits.argmax(dim=-1)
    if torch.distributed.is_initialized():
        # Trainer pads time steps across workers with -100 before this hook.
        frame_min, frame_max = torch.aminmax(logits, dim=-1)
        prediction_ids = prediction_ids.masked_fill(
            (frame_min == -100) & (frame_max == -100), -100
        )
    return prediction_ids


def compute_metrics(prediction, references):
    pad_id = processor.tokenizer.pad_token_id
    prediction_ids = np.where(prediction.predictions == -100, pad_id, prediction.predictions)
    prediction_texts = processor.tokenizer.batch_decode(prediction_ids)

    if TASK == "letter":
        references = ["".join(text.lower().split()) for text in references]
        prediction_texts = ["".join(text.lower().split()) for text in prediction_texts]
    elif TASK == "pinyin":
        references = [text.lower() for text in references]

    metric = {"cer": cer, "wer": wer}[metric_name]
    return {metric_name: metric(references, prediction_texts)}

#### __➕ Laod the XLS-R Model__

In [17]:
# laod the XLS-R model
CHECKPOINT = "facebook/wav2vec2-xls-r-300m"

def build_model(processor):
    model = Wav2Vec2ForCTC.from_pretrained(
        CHECKPOINT,
        attention_dropout=0.0,
        hidden_dropout=0.0,
        feat_proj_dropout=0.0,
        mask_time_prob=0.05,
        layerdrop=0.0,
        ctc_loss_reduction="mean",
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
        ignore_mismatched_sizes=True,
    )
    # freeze the feature encoder to ensure the feature not updated during back prop.
    model.freeze_feature_encoder()
    return model

In [18]:
# smoke_eval_indices = range(4)
# smoke_train_dataset = training_dataset["train"].select(range(8))
# smoke_eval_dataset = training_dataset["validation"].select(smoke_eval_indices)
# smoke_references = list(dataset["validation"].select(smoke_eval_indices)[text_column])

# print(smoke_train_dataset)
# print(smoke_eval_dataset)

In [19]:
# smoke_output_dir = MODEL_OUTPUT_DIR / TASK / "smoke_test"

# smoke_args = TrainingArguments(
#     output_dir=str(smoke_output_dir),

#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=2,

#     max_steps=2,
#     learning_rate=3e-4,
#     warmup_steps=0,

#     gradient_checkpointing=True,
#     fp16=torch.cuda.is_available(),

#     eval_strategy="steps",
#     eval_steps=1,

#     save_strategy="no",
#     logging_strategy="steps",
#     logging_steps=1,

#     remove_unused_columns=False,
#     report_to="none",
# )

In [20]:
# set_seed(smoke_args.seed)
# smoke_trainer = Trainer(
#     model=build_model(processor),
#     args=smoke_args,
#     data_collator=data_collator,
#     train_dataset=smoke_train_dataset,
#     eval_dataset=smoke_eval_dataset,
#     processing_class=processor,
#     compute_metrics=partial(compute_metrics, references=smoke_references),
#     preprocess_logits_for_metrics=preprocess_logits_for_metrics,
# )

In [21]:
# smoke_result = smoke_trainer.train()
# print(smoke_result)

# smoke_metrics = smoke_trainer.evaluate()
# print(smoke_metrics)

# del smoke_trainer

#### __✖️ Formal Trainer__

In [22]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Selected device:", device)

Selected device: cuda


In [23]:
OUTPUT_DIR = MODEL_OUTPUT_DIR / TASK / "finetune_v2"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=3,

    per_device_train_batch_size=3,
    # keep variable-length audio unpadded when decoding evaluation outputs.
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=3e-4,
    warmup_steps=500,

    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    
    eval_strategy="steps",
    eval_steps=400,
    save_strategy="best",
    logging_steps=400,

    # save the best model
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    greater_is_better=False,

    save_total_limit=1,
    save_only_model=False,
    eval_accumulation_steps=10,

    remove_unused_columns=False,
    seed=42,
)

c:\Users\liudr\anaconda3\envs\liushp02\Lib\site-packages\transformers\training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


In [24]:
validation_references = list(dataset["validation"][text_column])
set_seed(training_args.seed)

trainer = Trainer(
    model=build_model(processor),
    args=training_args,
    data_collator=data_collator,
    train_dataset=training_dataset["train"],
    eval_dataset=training_dataset["validation"],
    processing_class=processor,
    compute_metrics=partial(compute_metrics, references=validation_references),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

Loading weights: 100%|██████████| 422/422 [00:00<00:00, 34320.21it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
assert any(p.requires_grad for p in trainer.model.wav2vec2.encoder.parameters())

In [27]:
train_result = trainer.train()
print(train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 4016, 'bos_token_id': 4015}.


Step,Training Loss,Validation Loss,Cer
400,303.573535,6.884029,1.000000
800,53.159663,6.434926,0.985114
1200,46.443457,4.519797,0.790470
1600,33.508857,3.457696,0.681412
2000,27.706997,2.871855,0.606575
2400,24.183979,2.560651,0.562952
2800,21.515737,2.221622,0.509151
3200,19.512369,1.981975,0.463214
3600,18.604052,1.814665,0.434341
4000,17.153524,1.681629,0.411046


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]


TrainOutput(global_step=7119, training_loss=37.25402772016347, metrics={'train_runtime': 14358.8585, 'train_samples_per_second': 11.895, 'train_steps_per_second': 0.496, 'total_flos': 2.5203104086939464e+19, 'train_loss': 37.25402772016347, 'epoch': 3.0})


In [28]:
model = trainer.model
tokenizer = processor.tokenizer
blank_id = model.config.pad_token_id
assert blank_id == tokenizer.pad_token_id
device = next(model.parameters()).device

print("step:", trainer.state.global_step, "eval strategy:", trainer.args.eval_strategy)
print("recent logs:", trainer.state.log_history[-4:])
print("trainable backbone tensors:", sum(p.requires_grad for p in model.wav2vec2.parameters()))

was_training = model.training
model.eval()
try:
    with torch.inference_mode():
        for split in ("train", "validation"):
            refs, preds = [], []
            for i in range(4):
                example = training_dataset[split][i]
                ref = dataset[split][text_column][i]
                label = tokenizer.decode(example["labels"], group_tokens=False)

                batch = data_collator([example])
                inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
                ids = model(**inputs).logits[0].argmax(dim=-1).cpu()
                pred = tokenizer.decode(ids.tolist())
                blank_ratio = (ids == blank_id).float().mean().item()

                print(f"{split}[{i}] blank={blank_ratio:.1%} | pred={pred!r} | ref={ref!r}")
                if label != ref:
                    print(f"  decoded label={label!r}")
                refs.append(ref)
                preds.append(pred)
            print(f"{split} sample CER={cer(refs, preds):.3f}")
finally:
    model.train(was_training)

step: 7119 eval strategy: IntervalStrategy.STEPS
recent logs: [{'loss': 10.979661865234375, 'grad_norm': 10.818975448608398, 'learning_rate': 1.4503701465478168e-05, 'epoch': 2.865799041045366, 'step': 6800}, {'eval_loss': 1.068301796913147, 'eval_cer': 0.2779448382839182, 'eval_runtime': 186.6503, 'eval_samples_per_second': 33.898, 'eval_steps_per_second': 33.898, 'epoch': 2.865799041045366, 'step': 6800}, {'eval_loss': 1.0570734739303589, 'eval_cer': 0.27477446831671043, 'eval_runtime': 185.7775, 'eval_samples_per_second': 34.057, 'eval_steps_per_second': 34.057, 'epoch': 3.0, 'step': 7119}, {'train_runtime': 14358.8585, 'train_samples_per_second': 11.895, 'train_steps_per_second': 0.496, 'total_flos': 2.5203104086939464e+19, 'train_loss': 37.25402772016347, 'epoch': 3.0, 'step': 7119}]
trainable backbone tensors: 394
train[0] blank=93.1% | pred='但这一点已经足构表名券福的超高人器' | ref='但这一点已经足够表明卷福的超高人气'
train[1] blank=93.3% | pred='重回世界顶间强对行业' | ref='重回世界顶尖强队行列'
train[2] blank=93.3% | pred='因为想米不


step: 2082 eval strategy: IntervalStrategy.STEPS
recent logs: [{'loss': 54.893681640625, 'grad_norm': 2.6619012355804443, 'learning_rate': 0.0002501888502794984, 'epoch': 0.6744296327519891, 'step': 1600}, {'eval_loss': 9.264058113098145, 'eval_cer': 1.0, 'eval_runtime': 194.9685, 'eval_samples_per_second': 32.451, 'eval_steps_per_second': 32.451, 'epoch': 0.6744296327519891, 'step': 1600}, {'loss': 54.454716796875, 'grad_norm': 2.778933525085449, 'learning_rate': 0.0002320592234476507, 'epoch': 0.8430370409399863, 'step': 2000}, {'eval_loss': 9.173480987548828, 'eval_cer': 1.0, 'eval_runtime': 164.6678, 'eval_samples_per_second': 38.423, 'eval_steps_per_second': 38.423, 'epoch': 0.8430370409399863, 'step': 2000}]
trainable backbone tensors: 0
train[0] blank=100.0% | pred='' | ref='但这一点已经足够表明卷福的超高人气'
train[1] blank=100.0% | pred='' | ref='重回世界顶尖强队行列'
train[2] blank=100.0% | pred='' | ref='因为小米不会做家装'
train[3] blank=100.0% | pred='' | ref='打开一首歌徐良的歌'
train sample CER=1.000
validation[0] blank=100.0% | pred='' | ref='中联航将被削减百分之十的飞行小时数'
validation[1] blank=100.0% | pred='' | ref='交易所要求需逐笔填列'
validation[2] blank=100.0% | pred='' | ref='七百零八点零九'
validation[3] blank=100.0% | pred='' | ref='得到借款五百八十七万元'
validation sample CER=1.000

In [29]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation score:", trainer.state.best_metric)

Best checkpoint: d:\shuangpingpho\semester-project-chinese-ASR\model_outputs\hanzi\finetune_v2\checkpoint-7119
Best validation score: 0.27477446831671043


In [ ]:
validation_metric_fn = trainer.compute_metrics
try:
    trainer.compute_metrics = partial(
        compute_metrics, references=list(dataset["test"][text_column])
    )
    test_metrics = trainer.evaluate(
        eval_dataset=training_dataset["test"],
        metric_key_prefix="test",
    )
finally:
    trainer.compute_metrics = validation_metric_fn

print(test_metrics)

In [ ]:
trainer.log_metrics(
    "test",
    test_metrics,
)

trainer.save_metrics(
    "test",
    test_metrics,
)